# Retrieval-Augmented Generation (RAG)

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [4]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")


In [5]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

/home/vscode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:06<00:00, 12.4MiB/s]


[('calories_per_100g', 89.0), ('food_category', 'fruits'), ('food_item', 'banana'), ('keywords', 'banana_fruits'), ('kj_per_100g', 374.0), ('serving_info', '100g')]
Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


[('calories_per_100g', 50.0), ('food_category', '(fruit)juices'), ('food_item', 'banana juice'), ('keywords', 'banana_juice_(fruit)juices'), ('kj_per_100g', 210.0), ('serving_info', '100ml')]
Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [8]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [ ]:
#calorie_lookup_tool('bananas')

'Nutrition Information:\nBanana (Fruits): 89.0 calories per 100g\nBanana Juice ((Fruit)Juices): 50.0 calories per 100g\nBanana Nut Bread (Pastries,Breads&Rolls): 326.0 calories per 100g'

In [12]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    """,
    tools=[calorie_lookup_tool]
)

In [ ]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result.final_output)

- Per 100 g: Banana 89 kcal; Apple 52 kcal.
- Rough total (typical sizes): Banana (~118 g) ≈ 105 kcal; Apple (~182 g) ≈ 95 kcal; total ≈ 200 kcal.


In [14]:
#new code to support multiple tools
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

In [16]:
qna_results = nutrition_qna.query(query_texts=["what are the two most common complications with pregnancy"], n_results=2)
for i, doc in enumerate(qna_results["documents"][0]):
    print(sorted(qna_results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

[('is_pregnancy', True)]
Question: What are some potential complications that could make a pregnancy high-risk?
        Answer: There are several factors that could lead to a high-risk pregnancy, such as anemia during pregnancy, hypertensive disorders of pregnancy, diabetes mellitus during pregnancy, obesity during pregnancy, and adolescent pregnancy.

        This Q&A pair provides information about nutrition and health topics.


[('is_pregnancy', True)]
Question: Which health problems can result in a more hazardous pregnancy?
        Answer: High risk pregnancies may be the outcome of numerous challenges such as anemia during gestation, high blood pressure complications associated with being pregnant, diabetes combined with pregnancy, obesity throughout pregnancy, and teenage motherhood.

        This Q&A pair provides information about nutrition and health topics.




In [ ]:
@function_tool
def qna_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up pregnancy and nutrition information.

    Args:
        query: The pregnancy related item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the response.
    """

    results = nutrition_qna.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        formatted_results.append(doc)

    return "Related answers to your question:\n" + "\n".join(formatted_results)

In [25]:
qna_lookup_tool("what are the two most common complications with pregnancy",3)

'Related answers to your question:\nQuestion: What are some potential complications that could make a pregnancy high-risk?\n        Answer: There are several factors that could lead to a high-risk pregnancy, such as anemia during pregnancy, hypertensive disorders of pregnancy, diabetes mellitus during pregnancy, obesity during pregnancy, and adolescent pregnancy.\n\n        This Q&A pair provides information about nutrition and health topics.\nQuestion: Which health problems can result in a more hazardous pregnancy?\n        Answer: High risk pregnancies may be the outcome of numerous challenges such as anemia during gestation, high blood pressure complications associated with being pregnant, diabetes combined with pregnancy, obesity throughout pregnancy, and teenage motherhood.\n\n        This Q&A pair provides information about nutrition and health topics.\nQuestion: What medical conditions can make a pregnancy more dangerous?\n        Answer: High risk pregnancies may happen due to 

In [41]:
calorie_agent2 = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    If are asked a question about nutrition or pregnancy, always use the qna_lookup_tool first to see if there is an answer in the knowledge base
    """,
    tools=[calorie_lookup_tool, qna_lookup_tool],
)

In [38]:
with trace("Nutrition Assistant with multiple tools"):
    result2 = await Runner.run(
        calorie_agent2,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result2.final_output)


Approximately 200 kcal total for a medium banana (about 118 g) and a medium apple (about 182 g).

Calories per 100 g:
- Banana: ~89 kcal/100 g
- Apple: ~52 kcal/100 g

Note: actual values vary with size.


In [43]:
with trace("Nutrition Assistant with multiple tools2"):
    result2 = await Runner.run(
        calorie_agent2,
        "What are the best meal choices for pregnant women",
    )
    print(result2.final_output)

Here are core healthy meal choices for pregnant women:

- Whole grains: cereals, millets, brown rice, oats
- Protein: dairy or fortified dairy alternatives, eggs, lean meats, fish (low-mercury), beans, lentils, tofu
- Fruits and vegetables: plenty of leafy greens (spinach, kale), vitamin C–rich fruits (oranges, berries, kiwis), a variety of vegetables
- Calcium sources: milk/yogurt/cheese or fortified alternatives
- Iron-rich foods: lean meat, beans, lentils, fortified cereals; pair with vitamin C foods to boost absorption
- Healthy fats: nuts, seeds, olive oil, avocado
- Hydration: water regularly; limit caffeine; avoid alcohol

Note: Always follow your healthcare provider’s guidance and consider a prenatal vitamin as recommended.
